# Feedlot Simulator V2 · Stage 6B.1
Benchmark sintetis CPU Google Colab dengan audit independen penuh. Kalender bisnis Edmonton
mencakup 1 Januari 2024 sampai 31 Desember 2025, termasuk carry-in, carry-out dan penutupan keuangan.

Pada menu **Runtime → Change runtime type**, pilih **Python 3** dan **Hardware accelerator → None**.
Sambungkan runtime CPU dengan jatah akun yang tersedia. Mulai dari **250**. Tekan ▶ pada setiap
sel dari atas ke bawah. Pada **Run Simulation**, centang kotak mulai dan masukkan jumlah placement
saat diminta. Membuka notebook tidak memulai simulasi.

Di iPhone, buka tautan Colab lewat Safari atau Chrome. Jika menu runtime atau formulir sulit
digunakan, aktifkan **Minta Situs Web Desktop** atau gunakan browser komputer.

## Check Environment
Periksa CPU, RAM, disk kosong dan versi Python sebelum menyiapkan sumber.

In [ ]:
# @title Check Environment
from datetime import date, datetime, timedelta, timezone
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import uuid
from zoneinfo import ZoneInfo

REPOSITORY = "https://github.com/wijaya-rama/portfolio-feedlot-management.git"
REVIEWED_SHA = "39770fb9e2867127f1ddac671ac694997e26b864"
SEED = 42
GIB = 1024**3
PINS = ("numpy==2.3.5", "pyarrow==19.0.1", "tzdata==2025.2")
CONFIG_DIR = "simulator_v2/stage6/configs"
HEALTH_VERSION = "stage6b1-baseline-health.1"
SUPPLY_POLICY = "material_specific"
DATA_REPORTS = {
    "config.json", "diagnostics.json", "monthly_summary.json", "validation.json", "weather.json",
}
REQUIRED_GATES = {
    "balanced_paired_entries", "balanced_two_line_journal", "biological_reserve_balance",
    "financial_close_publication", "warehouse_replay_publication", "warehouse_af_value",
    "warehouse_dm", "stock_equation", "period_end_stock_reporting", "final_simulation_stock_reporting",
    "snapshot_publication", "stable_original_snapshot", "stable_restated_snapshot",
}
# Batas awal untuk ruang kerja, staging SQLite, dan salinan arsip. Satuan GiB.
RESOURCE_FLOORS = {250: (4, 6, 1), 500: (5, 8, 2), 1500: (8, 16, 6)}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def read_json(path):
    return json.loads(path.read_bytes())


def sha256_path(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def available_ram():
    line = next(row for row in Path("/proc/meminfo").read_text().splitlines()
                if row.startswith("MemAvailable:"))
    available = int(line.split()[1]) * 1024
    for limit_path, current_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        limit_path, current_path = Path(limit_path), Path(current_path)
        if limit_path.is_file() and current_path.is_file():
            limit = limit_path.read_text().strip()
            if limit.isdecimal():
                available = min(available, max(0, int(limit) - int(current_path.read_text())))
    return available


def environment(root):
    return {
        "python": platform.python_version(),
        "system": platform.system(),
        "architecture": platform.machine(),
        "logical_cpus": len(os.sched_getaffinity(0)),
        "ram_available_bytes": available_ram(),
        "local_disk_free_bytes": shutil.disk_usage(root).free,
    }


def display_environment(info):
    print(f"Python {info['python']} | {info['system']} {info['architecture']}")
    print(f"CPU logis  {info['logical_cpus']} | RAM tersedia  {info['ram_available_bytes'] / GIB:.2f} GiB")
    print(f"Disk lokal kosong  {info['local_disk_free_bytes'] / GIB:.2f} GiB")


def check_runtime(root):
    require(platform.system() == "Linux" and platform.machine() == "x86_64",
            "Pilih runtime Linux x86_64 di Google Colab.")
    require(platform.python_implementation() == "CPython" and (3, 11) <= sys.version_info[:2] <= (3, 13),
            "Versi runtime perlu CPython 3.11–3.13. Pilih versi runtime Colab yang kompatibel.")
    require(not any(Path("/dev").glob("nvidia[0-9]*")) and not any(Path("/dev").glob("accel*"))
            and not any(os.environ.get(key) for key in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_WORKER_HOSTNAMES")),
            "Pilih Hardware accelerator None (CPU).")
    require(shutil.which("git") and shutil.disk_usage(root).free >= 2 * GIB,
            "Git dan setidaknya 2 GiB disk lokal diperlukan untuk menyiapkan runtime.")


def runner_env():
    env = os.environ.copy()
    for key in ("PYTHONPATH", "PYTHONHOME"):
        env.pop(key, None)
    env.update(PYTHONNOUSERSITE="1", PYTHONDONTWRITEBYTECODE="1", PYTHONHASHSEED="0", PYTHONTZPATH="")
    return env


LOCAL_ROOT = Path("/content")
require(LOCAL_ROOT.is_dir(), "Buka notebook di Google Colab agar /content tersedia.")
display_environment(environment(LOCAL_ROOT))
check_runtime(LOCAL_ROOT)
SESSION = Path(tempfile.mkdtemp(prefix="feedlot-stage6b1-", dir=LOCAL_ROOT))
SOURCE = SESSION / "reviewed-source"
PYTHON = SESSION / "venv" / "bin" / "python"
ATTEMPT = None

## Get Verified Source
Ambil commit review publik dari GitHub dan periksa SHA checkout. Langkah ini tidak memerlukan kredensial GitHub.

In [ ]:
# @title Get Verified Source
def git(source, *args):
    return subprocess.check_output(
        ["git", "-C", str(source), *args], text=True,
        env={**runner_env(), "GIT_TERMINAL_PROMPT": "0"},
    ).strip()


def verify_source(source):
    require(git(source, "rev-parse", "HEAD") == REVIEWED_SHA,
            "SHA kode simulator berbeda dari commit review.")
    require(not git(source, "status", "--porcelain", "--untracked-files=all"),
            "Checkout simulator berubah. Mulai sesi baru untuk mengambil kode bersih.")


def get_source(source):
    if not source.joinpath(".git").is_dir():
        source.mkdir(exist_ok=True)
        git(source, "init", "--quiet")
    if subprocess.run(["git", "-C", str(source), "cat-file", "-e", f"{REVIEWED_SHA}^{{commit}}"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
        git(source, "fetch", "--no-tags", "--depth=1", REPOSITORY, REVIEWED_SHA)
        require(git(source, "rev-parse", "FETCH_HEAD") == REVIEWED_SHA, "SHA dari GitHub tidak cocok.")
    if subprocess.run(["git", "-C", str(source), "rev-parse", "--verify", "HEAD"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
        git(source, "checkout", "--detach", "--quiet", REVIEWED_SHA)
    verify_source(source)


check_runtime(SESSION)
get_source(SOURCE)
print(f"Commit review terverifikasi  {REVIEWED_SHA}")

## Install Dependencies
Buat lingkungan Python lokal terisolasi. Verifikasi versi paket, codec Zstandard, zona waktu Edmonton
dan impor runner sebelum menjalankan simulasi.

In [ ]:
# @title Install Dependencies
def check_dependencies(python, source):
    script = """
from importlib.metadata import version
import sys
from zoneinfo import ZoneInfo
import numpy
import pyarrow as pa
import pyarrow.parquet
from simulator_v2.stage6.__main__ import build
assert all(version(name) == expected for name, expected in (pin.split('==') for pin in sys.argv[1:]))
assert pa.Codec.is_available('zstd')
assert ZoneInfo('America/Edmonton')
"""
    subprocess.run([str(python), "-c", script, *PINS], cwd=source, env=runner_env(), check=True)


def install_dependencies(python, source):
    subprocess.run([sys.executable, "-m", "venv", str(python.parents[1])], check=True)
    subprocess.run([str(python), "-m", "pip", "install", "--disable-pip-version-check",
                    "--no-cache-dir", "--only-binary=:all:", *PINS], check=True)
    subprocess.run([str(python), "-m", "pip", "check"], check=True)
    check_dependencies(python, source)


verify_source(SOURCE)
install_dependencies(PYTHON, SOURCE)
print("Dependensi runtime siap ", ", ".join(PINS))

## Connect Google Drive
Izinkan Colab mengakses Google Drive saat diminta. Simulator bekerja di disk lokal Colab.
Drive menyimpan status, laporan, dan satu arsip dataset setelah validasi.

In [ ]:
# @title Connect Google Drive
from google.colab import drive

MOUNT = LOCAL_ROOT / "drive"
drive.mount(str(MOUNT))
require(MOUNT.joinpath("MyDrive").is_dir(), "Google Drive belum terhubung.")
DRIVE_ROOT = MOUNT / "MyDrive" / "Feedlot_V2_Benchmarks"
DRIVE_ROOT.mkdir(exist_ok=True)
print("Folder hasil di Drive: My Drive / Feedlot_V2_Benchmarks")

## Select Simulation Size
Pilih **250** untuk run pertama. Pilih **500** setelah 250 berstatus `completed` di Drive
dan sumber daya mencukupi. Pilihan **1.500** memakai konfigurasi kandidat
`stage6b1-baseline-1500-candidate.1` dari jadwal Stage 5 dan algoritme skala Stage 6B.
Status pilot saat ini **HOLD**. Centang kedua kotak pilot hanya setelah memberi persetujuan
eksplisit dan menuntaskan gerbang kesiapan yang tercatat, termasuk hasil 250 dan 500,
evaluasi holdout 7/30/60 hari dengan lima seed, dan penerimaan kapasitas, overflow hospital,
serta segregasi lot. Batas sumber daya yang diproyeksikan hanya untuk perencanaan.
Ukuran 15.000 disimpan untuk rilis terpisah dengan persetujuan eksplisit.

[Gerbang kesiapan pilot](https://github.com/wijaya-rama/portfolio-feedlot-management/blob/39770fb9e2867127f1ddac671ac694997e26b864/simulator_v2/stage6/PILOT_READINESS.md)

In [ ]:
# @title Select Simulation Size
Placements = "250" # @param ["250", "500", "1500"]
Saya_menyetujui_pilot_1500 = False # @param {type:"boolean"}
Gerbang_kesiapan_pilot_selesai = False # @param {type:"boolean"}


def pilot_overrides():
    return json.loads(r'''
{
  "version": "stage6b1-baseline-1500-candidate.1",
  "placements": [
    {"date": "2024-01-01", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-01-13", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-01-25", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-02-06", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-02-18", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-03-01", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-03-13", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-03-25", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-04-06", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-04-18", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-04-30", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-05-12", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-05-24", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-06-05", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-06-17", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-06-29", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-07-11", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-07-23", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-08-04", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-08-16", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-08-28", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-09-09", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-09-21", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-10-03", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-10-15", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-10-27", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-11-08", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-11-20", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2024-12-02", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2024-12-14", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2024-12-26", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-01-07", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-01-19", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-01-31", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-02-12", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-02-24", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-03-08", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-03-20", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-04-01", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-04-13", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-04-25", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-05-07", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-05-19", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-05-31", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-06-12", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-06-24", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-07-06", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-07-18", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-07-30", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-08-11", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-08-23", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-09-04", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-09-16", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-09-28", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-10-10", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-10-22", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-11-03", "head": 25, "mean_kg": 374, "minimum_dof": 195},
    {"date": "2025-11-15", "head": 25, "mean_kg": 352, "minimum_dof": 155},
    {"date": "2025-11-27", "head": 25, "mean_kg": 363, "minimum_dof": 175},
    {"date": "2025-12-09", "head": 25, "mean_kg": 374, "minimum_dof": 195}
  ],
  "carry_in": [
    {"head": 25, "location": "SYN-P01", "weight_kg": 506.0, "entry_age_days": 459, "days_on_feed": 149, "minimum_dof": 155, "wip_cents": 219090, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-01"},
    {"head": 25, "location": "SYN-P02", "weight_kg": 513.7, "entry_age_days": 467, "days_on_feed": 157, "minimum_dof": 175, "wip_cents": 222370, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-02"},
    {"head": 25, "location": "SYN-P03", "weight_kg": 521.4, "entry_age_days": 475, "days_on_feed": 165, "minimum_dof": 195, "wip_cents": 225650, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-03"},
    {"head": 25, "location": "SYN-P04", "weight_kg": 471.5, "entry_age_days": 423, "days_on_feed": 113, "minimum_dof": 155, "wip_cents": 204330, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-04"},
    {"head": 25, "location": "SYN-P05", "weight_kg": 479.2, "entry_age_days": 431, "days_on_feed": 121, "minimum_dof": 175, "wip_cents": 207610, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-05"},
    {"head": 25, "location": "SYN-P06", "weight_kg": 486.8, "entry_age_days": 439, "days_on_feed": 129, "minimum_dof": 195, "wip_cents": 210890, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-06"},
    {"head": 25, "location": "SYN-P01", "weight_kg": 436.9, "entry_age_days": 387, "days_on_feed": 77, "minimum_dof": 155, "wip_cents": 189570, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-07"},
    {"head": 25, "location": "SYN-P02", "weight_kg": 444.6, "entry_age_days": 395, "days_on_feed": 85, "minimum_dof": 175, "wip_cents": 192850, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-08"},
    {"head": 25, "location": "SYN-P03", "weight_kg": 452.3, "entry_age_days": 403, "days_on_feed": 93, "minimum_dof": 195, "wip_cents": 196130, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-09"},
    {"head": 25, "location": "SYN-P04", "weight_kg": 402.4, "entry_age_days": 351, "days_on_feed": 41, "minimum_dof": 155, "wip_cents": 174810, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-10"},
    {"head": 25, "location": "SYN-P05", "weight_kg": 410.0, "entry_age_days": 359, "days_on_feed": 49, "minimum_dof": 175, "wip_cents": 178090, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-11"},
    {"head": 25, "location": "SYN-P06", "weight_kg": 417.7, "entry_age_days": 367, "days_on_feed": 57, "minimum_dof": 195, "wip_cents": 181370, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-12"},
    {"head": 25, "location": "SYN-P01", "weight_kg": 367.8, "entry_age_days": 315, "days_on_feed": 5, "minimum_dof": 155, "wip_cents": 160050, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-13"},
    {"head": 25, "location": "SYN-P02", "weight_kg": 394.7, "entry_age_days": 343, "days_on_feed": 33, "minimum_dof": 195, "wip_cents": 171530, "status": "active", "source": "SYN-PREFLIGHT-OPEN-2024-14"}
  ],
  "capacities": {
    "receiving": [
      25,
      25
    ],
    "production": [
      90,
      90,
      90,
      90,
      90,
      90
    ],
    "hospital": [
      16,
      16
    ]
  },
  "shipment": {
    "weekday": 1,
    "minimum_dof": 175,
    "threshold_kg": 490,
    "lot_fraction": 0.65,
    "maximum_head": 75,
    "grade_missing_probability": 0.14,
    "settlement_lag_days": [
      2,
      6
    ],
    "grid_cents_per_kg": 800,
    "freight_cents": 4250
  }
}
''')

def config_for(source, size):
    require(size in RESOURCE_FLOORS, "Ukuran yang tersedia adalah 250, 500, dan 1.500.")
    config = read_json(source / CONFIG_DIR / f"stage6b1-baseline-{min(size, 500)}.json")
    if size == 1500:
        config.update(pilot_overrides())
    version = "stage6b1-baseline-1500-candidate.1" if size == 1500 else f"stage6b1-baseline-{size}.1"
    require(config["version"] == version and config["seed"] == SEED and (config["start"], config["end"]) == ("2024-01-01", "2025-12-31")
            and sum(item["head"] for item in config["placements"]) == size
            and bool(config["carry_in"]) and config["health"]["scenario_version"] == HEALTH_VERSION,
            "Konfigurasi tidak cocok dengan ukuran, seed, kalender, atau skenario baseline.")
    warehouse = config["warehouse"]["ingredients"]
    keys = ("opening_grams_per_design_head", "ordered_grams_per_design_head",
            "reorder_grams_per_design_head", "bin_grams_per_design_head")
    require({item["code"] for item in warehouse} == {"SILAGE", "BARLEY", "OATS"}
            and all(len({item[key] for item in warehouse}) == 3 for key in keys),
            "Kebijakan persediaan per bahan tidak lengkap.")
    return config


def previous_result(root, size, version):
    candidates = []
    for marker in root.glob("*/status.json"):
        try:
            saved = read_json(marker)
            if (saved["status"], saved["code_sha"], saved["placements"], saved["seed"],
                    saved["config_version"]) != ("completed", REVIEWED_SHA, size, SEED, version):
                continue
            package = marker.parent / "dataset.tar"
            report = marker.parent / "validation.json"
            if (package.stat().st_size == saved["package_bytes"]
                    and sha256_path(report) == saved["saved_sha256"]["validation.json"]):
                verify_audit(read_json(report))
                candidates.append(saved)
        except (OSError, ValueError, KeyError, TypeError, RuntimeError):
            continue
    require(candidates, f"Selesaikan dan simpan benchmark {size} dengan commit review ini terlebih dahulu.")
    return max(candidates, key=lambda item: item["updated_at"])


def require_resources(size, local, drive_root, previous=None):
    ram, local_disk, durable_disk = (amount * GIB for amount in RESOURCE_FLOORS[size])
    if previous:
        scale = size / previous["placements"]
        peak = previous["performance"]["peak_rss_kib_linux"] * 1024
        projected_archive = previous["package_bytes"] * scale
        ram = max(ram, peak * scale * 1.25)
        local_disk = max(local_disk, projected_archive * 3 + GIB)
        durable_disk = max(durable_disk, projected_archive * 1.25)
    for label, free, limit in (
        ("RAM tersedia", available_ram(), ram),
        ("Disk lokal kosong", shutil.disk_usage(local).free, local_disk),
        ("Ruang Drive yang dilaporkan", shutil.disk_usage(drive_root).free, durable_disk),
    ):
        print(f"{label}: {free / GIB:.2f} GiB | batas untuk run ini: {limit / GIB:.2f} GiB")
        require(free >= limit, f"{label} belum cukup untuk {size} placement.")


def verify_audit(validation):
    require(validation.get("mode") == "full" and validation.get("complete_independent_audit") is True,
            "Audit independen penuh belum lulus. Mode incremental menghasilkan audit yang belum lengkap.")
    gates = validation["hard_gates"]
    require(REQUIRED_GATES <= gates.keys(), "Laporan audit kehilangan gerbang wajib.")
    require(all(row["passed"] is True for row in gates.values())
            and all(gates[name]["assertions"] > 0 for name in REQUIRED_GATES),
            "Audit memuat gerbang gagal atau gerbang wajib yang belum dijalankan.")
    for category in ("movement", "financial"):
        checks = validation[category]
        require(checks and all(check["failed_count"] == 0 for check in checks),
                f"Kontrol {category} kosong atau gagal.")
    return len(gates)


SIZE = int(Placements)
print(f"Ukuran dipilih {SIZE:,} placement | seed {SEED} | {HEALTH_VERSION}".replace(",", "."))

## Run Simulation
Centang **Mulai_simulasi**, tekan ▶, lalu ketik ukuran yang dipilih. Setiap percobaan
memakai folder baru. Jalankan satu percobaan pada satu waktu. Runtime dapat dihentikan oleh
Colab dan durasinya bergantung pada mesin. Target referensi 600 detik adalah target performa.

In [ ]:
# @title Run Simulation
Mulai_simulasi = False # @param {type:"boolean"}


def write_status(attempt, state, **fields):
    document = {**attempt["status"], **fields, "status": state,
                "updated_at": datetime.now(timezone.utc).isoformat()}
    target = attempt["durable"] / "status.json"
    temporary = target.with_suffix(".json.partial")
    temporary.write_text(json.dumps(document, indent=2) + "\n", encoding="utf-8")
    temporary.replace(target)
    attempt["status"] = document


def record_failure(attempt, state, error, **fields):
    attempt["status"]["status"] = state
    attempt["status"]["error"] = f"{type(error).__name__}: {error}"
    try:
        write_status(attempt, state, **fields)
        log = attempt["work"] / "runner.log"
        if log.is_file():
            shutil.copyfile(log, attempt["durable"] / "runner.log")
        if state == "validation_failed":
            for name in ("run_manifest.json", "hashes.json", "validation.json", "performance.json"):
                report = attempt["work"] / "dataset" / name
                if report.is_file():
                    shutil.copyfile(report, attempt["durable"] / f"unverified-{name}")
    except OSError as save_error:
        print(f"Pencatatan di Drive terhenti ({save_error}). Periksa status.json sebelum menggunakan hasil.")


def command_for(python, config, dataset):
    return [str(python), "-u", "-m", "simulator_v2.stage6", "--config", str(config),
            "--seed", str(SEED), "--output", str(dataset)]


def start_attempt(session, drive_root, python, config, size):
    info = environment(session)
    attempt_id = f"placements-{size}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:8]}"
    work, durable = session / attempt_id, drive_root / attempt_id
    config_path = work / "selected-config.json"
    command = command_for(python, config_path, work / "dataset")
    attempt = {"work": work, "durable": durable, "config": config, "status": {
        "attempt_id": attempt_id, "code_sha": REVIEWED_SHA, "placements": size, "seed": SEED,
        "config_version": config["version"], "health_version": HEALTH_VERSION, "supply_policy": SUPPLY_POLICY,
        "started_at": datetime.now(timezone.utc).isoformat(), "resources": info, "dependencies": list(PINS),
        "command": command, "validation_status": "pending", "complete_independent_audit": False,
    }}
    durable.mkdir()
    try:
        write_status(attempt, "running")
        work.mkdir()
        config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
        shutil.copyfile(config_path, durable / config_path.name)
    except BaseException as error:
        record_failure(attempt, "setup_failed", error)
        raise
    return attempt


def run_simulation(attempt, source):
    started = time.monotonic()
    log = attempt["work"] / "runner.log"
    print(f"Run {attempt['status']['attempt_id']} sedang berjalan. Log disimpan di akhir fase.", flush=True)
    try:
        with log.open("wb") as output, subprocess.Popen(
            attempt["status"]["command"], cwd=source, env=runner_env(), stdout=output, stderr=subprocess.STDOUT
        ) as process:
            next_report = 60
            try:
                while True:
                    try:
                        code = process.wait(timeout=10)
                        break
                    except subprocess.TimeoutExpired:
                        elapsed = time.monotonic() - started
                        require(shutil.disk_usage(attempt["work"]).free >= GIB // 2,
                                "Disk lokal mendekati penuh. Proses dihentikan.")
                        if elapsed >= next_report:
                            print(f"Masih berjalan, {elapsed / 60:.0f} menit.", flush=True)
                            next_report += 60
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
        if code:
            print(log.read_text(errors="replace")[-3000:])
        require(code == 0, f"Runner berhenti dengan kode {code}. Lihat runner.log.")
        require((attempt["work"] / "dataset").is_dir(), "Runner belum memublikasikan dataset lokal.")
        write_status(attempt, "awaiting_validation", runner_return_code=code,
                     runner_wall_seconds=round(time.monotonic() - started, 3))
        print("Runner selesai. Jalankan Validate Results.")
    except BaseException as error:
        record_failure(attempt, "interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
                       error, validation_status="unverified",
                       runner_wall_seconds=round(time.monotonic() - started, 3))
        raise


def authorize_run(size, source, session, drive_root, pilot_approved, pilot_ready):
    verify_source(source)
    previous = None
    if size >= 500:
        previous = previous_result(drive_root, 250, "stage6b1-baseline-250.1")
    if size == 1500:
        require(pilot_approved and pilot_ready,
                "Pilot 1.500 masih HOLD sampai ada persetujuan dan semua gerbang kesiapan dipenuhi.")
        previous = previous_result(drive_root, 500, "stage6b1-baseline-500.1")
    require_resources(size, session, drive_root, previous)


if Mulai_simulasi:
    authorize_run(SIZE, SOURCE, SESSION, DRIVE_ROOT, Saya_menyetujui_pilot_1500, Gerbang_kesiapan_pilot_selesai)
    check_dependencies(PYTHON, SOURCE)
    chosen_config = config_for(SOURCE, SIZE)
    confirmation = input(f"Ketik {SIZE} untuk memulai run baru, lalu tekan Enter. ").strip()
    if confirmation == str(SIZE):
        ATTEMPT = start_attempt(SESSION, DRIVE_ROOT, PYTHON, chosen_config, SIZE)
        run_simulation(ATTEMPT, SOURCE)
    else:
        print("Permintaan mulai dibatalkan.")
else:
    print("Belum ada run baru. Centang Mulai_simulasi saat siap.")

## Validate Results
Runner melakukan audit penuh atas Parquet sebelum publikasi lokal. Sel ini memeriksa kelengkapan
audit, checksum partisi dan laporan, carry-in/carry-out, serta penutupan tiap bulan untuk kedua view.
Audit incremental mencakup sebagian pemeriksaan dan berstatus `complete_independent_audit=false`.

In [ ]:
# @title Validate Results
def validate_dataset(dataset, config):
    started = time.monotonic()
    manifest = read_json(dataset / "run_manifest.json")
    index = read_json(dataset / "hashes.json")
    require(manifest["code_checkout_commit_at_build"] == REVIEWED_SHA and manifest["synthetic"] is True
            and manifest["run_id"] == index["run_id"] and manifest["schema_version"] == index["schema_version"],
            "Manifest tidak cocok dengan kode review atau indeks hasil.")
    require(manifest["hashes_sha256"] == sha256_path(dataset / "hashes.json")
            and manifest["config_sha256"] == sha256_path(dataset / "config.json")
            and read_json(dataset / "config.json") == config,
            "Hash deterministik atau konfigurasi keluaran tidak cocok.")
    end = datetime.fromisoformat(manifest["effective_before"]).astimezone(ZoneInfo("America/Edmonton"))
    cutoff = datetime.fromisoformat(manifest["final_knowledge_cutoff"])
    require(end.date() == date.fromisoformat(config["end"]) + timedelta(days=1)
            and cutoff - end == timedelta(days=max(config["shipment"]["settlement_lag_days"]) + 4),
            "Horizon efektif atau cutoff penutupan tidak cocok.")

    partitions, reports = index["partitions"], index["artifacts"]
    require(bool(partitions) and set(reports) == DATA_REPORTS, "Indeks partisi atau laporan belum lengkap.")
    actual = {path.relative_to(dataset).as_posix() for path in dataset.rglob("*.parquet")}
    require(actual == partitions.keys(), "Inventaris Parquet berbeda dari indeks deterministik.")
    for name, row in partitions.items():
        path = dataset / name
        require(path.stat().st_size == row["bytes"] and sha256_path(path) == row["sha256"],
                f"Ukuran atau SHA-256 partisi berbeda ({name}).")
    for name, digest in reports.items():
        require(sha256_path(dataset / name) == digest, f"SHA-256 laporan berbeda ({name}).")

    validation = read_json(dataset / "validation.json")
    gate_count = verify_audit(validation)
    expected_months = set()
    month = date.fromisoformat(config["start"]).replace(day=1)
    final_date = date.fromisoformat(config["end"])
    while month <= final_date:
        expected_months.add(month.isoformat()[:7])
        month = (month.replace(day=28) + timedelta(days=4)).replace(day=1)
    closes = read_json(dataset / "monthly_summary.json")
    for view in ("as_known", "restated"):
        months = [row["month"] for row in closes if row["view"] == view]
        require(len(months) == len(expected_months) and set(months) == expected_months,
                f"Penutupan bulanan tidak lengkap untuk {view}.")
    require(len(closes) == 2 * len(expected_months), "Ada view atau penutupan tambahan yang tidak diharapkan.")

    diagnostics = read_json(dataset / "diagnostics.json")
    head = diagnostics["head"]
    stock = diagnostics["stock_reporting"]
    require(head["opening"] == sum(row["head"] for row in config["carry_in"])
            and stock["final_simulation_date"] == config["end"]
            and head["carry_out"] == stock["final_simulation_soh"]
            and head["opening"] + head["placements"] == head["sale"] + head["death"] + head["carry_out"],
            "Carry-in, carry-out, atau rekonsiliasi jumlah ternak berbeda.")
    performance = read_json(dataset / "performance.json")
    for field in ("wall_seconds_before_atomic_rename", "cross_domain_validation_seconds", "peak_rss_kib_linux"):
        require(math.isfinite(performance[field]) and performance[field] > 0, f"Metrik {field} belum lengkap.")
    require(performance["parquet_bytes"] == sum(row["bytes"] for row in partitions.values())
            and performance["parquet_partitions"] == len(partitions), "Metrik Parquet berbeda dari inventaris.")
    return {"run_id": manifest["run_id"], "hashes_sha256": manifest["hashes_sha256"],
            "model_version": manifest["model_version"], "schema_version": manifest["schema_version"],
            "gate_count": gate_count, "financial_closes": len(closes), "head": head, "performance": performance,
            "validation_mode": "full", "complete_independent_audit": True, "validation_status": "passed",
            "artifact_verification_seconds": round(time.monotonic() - started, 3)}


def validate_attempt(attempt):
    require(attempt["status"]["status"] == "awaiting_validation", "Run belum siap divalidasi.")
    try:
        result = validate_dataset(attempt["work"] / "dataset", attempt["config"])
        write_status(attempt, "validated", **result)
        print(f"Audit penuh lulus, {result['gate_count']} kategori dan {result['financial_closes']} penutupan. Jalankan Save Results.")
    except BaseException as error:
        record_failure(attempt, "validation_failed", error, validation_status="failed",
                       complete_independent_audit=False)
        raise


if ATTEMPT and ATTEMPT["status"]["status"] == "awaiting_validation":
    validate_attempt(ATTEMPT)
else:
    print("Tidak ada run baru yang menunggu validasi.")

## Save Results
Setelah status `validated`, simpan laporan dan paket `dataset.tar` ke Drive.
Gunakan paket dengan `status.json` bernilai `completed`. Status ditulis setelah salinan Drive
diverifikasi. Jika penyalinan gagal selama sesi masih hidup, periksa kuota atau sambungan Drive,
lalu ulangi **Save Results**.

Checkpoint dispatch harian dan SQLite digunakan di dalam satu build. Jika proses simulasi berhenti
atau runtime hilang, sambungkan runtime dan jalankan ulang dari awal dengan konfigurasi dan seed
yang sama. Folder lama tetap menjadi catatan percobaan. Abaikan paket `.partial`.

In [ ]:
# @title Save Results
def copy_verified(source, destination, expected=None):
    if expected is None:
        expected = sha256_path(source)
    temporary = destination.with_suffix(destination.suffix + ".partial")
    shutil.copyfile(source, temporary)
    require(temporary.stat().st_size == source.stat().st_size and sha256_path(temporary) == expected,
            f"Salinan Drive gagal diverifikasi ({destination.name}).")
    temporary.replace(destination)
    return expected


def make_package(dataset, archive):
    files = sorted(path for path in dataset.rglob("*") if path.is_file())
    reserve = sum(path.stat().st_size + 4096 for path in files) + 100 * 1024
    require(shutil.disk_usage(archive.parent).free >= reserve + GIB // 2,
            "Disk lokal belum cukup untuk arsip. Bebaskan ruang lalu ulangi Save Results.")
    temporary = archive.with_suffix(".tar.partial")
    with tarfile.open(temporary, "w", format=tarfile.PAX_FORMAT) as output:
        for path in files:
            output.add(path, arcname=(Path("dataset") / path.relative_to(dataset)).as_posix(), recursive=False)
    temporary.replace(archive)


def save_attempt(attempt):
    state = attempt["status"]
    require(state["status"] in {"validated", "save_failed"} and state["validation_status"] == "passed"
            and state["complete_independent_audit"] is True, "Penyimpanan menunggu audit penuh yang lulus.")
    started = time.monotonic()
    work, durable = attempt["work"], attempt["durable"]
    dataset, archive = work / "dataset", work / "dataset.tar"
    try:
        if not archive.is_file() or "local_archive_sha256" not in attempt:
            archive.with_suffix(".tar.partial").unlink(missing_ok=True)
            make_package(dataset, archive)
            attempt["local_archive_sha256"] = sha256_path(archive)
        require(shutil.disk_usage(durable).free >= archive.stat().st_size + 100 * 1024**2,
                "Ruang Drive yang dilaporkan belum cukup untuk menyimpan paket.")
        saved = {}
        for name in ("config.json", "run_manifest.json", "hashes.json", "validation.json", "performance.json"):
            saved[name] = copy_verified(dataset / name, durable / name)
        saved["runner.log"] = copy_verified(work / "runner.log", durable / "runner.log")
        saved["dataset.tar"] = copy_verified(archive, durable / archive.name, attempt["local_archive_sha256"])
        write_status(attempt, "completed", saved_sha256=saved, package_bytes=archive.stat().st_size,
                     save_seconds=round(time.monotonic() - started, 3), error=None)
        print(f"Paket terverifikasi tersimpan di My Drive / Feedlot_V2_Benchmarks / {durable.name}")
    except BaseException as error:
        record_failure(attempt, "save_failed", error)
        raise


if ATTEMPT and ATTEMPT["status"]["status"] in {"validated", "save_failed"}:
    save_attempt(ATTEMPT)
else:
    print("Paket baru akan disimpan sesudah audit penuh lulus.")

## View Summary
Periksa status akhir, durasi runner, waktu audit penuh, puncak RSS proses,
ukuran Parquet aktual dan lokasi paket tersimpan.

In [ ]:
# @title View Summary
def view_summary(attempt):
    status = read_json(attempt["durable"] / "status.json")
    print(f"Status tersimpan  {status['status']} | Audit  {status['validation_status']}")
    print(f"Placement diminta  {status['placements']} | Seed  {status['seed']} | Commit  {status['code_sha']}")
    if status.get("error"):
        print(status["error"])
    if status.get("validation_status") == "passed":
        measured = status["performance"]
        print("Pengukuran dari runtime CPU ini")
        print(f"Durasi runner  {status['runner_wall_seconds']:.2f} detik")
        print(f"Audit penuh  {measured['cross_domain_validation_seconds']:.2f} detik")
        print(f"Pemeriksaan integritas  {status['artifact_verification_seconds']:.2f} detik")
        print(f"Puncak RSS proses simulator  {measured['peak_rss_kib_linux'] * 1024 / GIB:.2f} GiB")
        print(f"Parquet  {measured['parquet_bytes'] / GIB:.2f} GiB | Penutupan  {status['financial_closes']}")
        print(f"Masuk aktual  {status['head']['placements']} | Carry-in  {status['head']['opening']} | Carry-out  {status['head']['carry_out']}")
    if status["status"] == "completed":
        print(f"Paket selesai  My Drive / Feedlot_V2_Benchmarks / {attempt['durable'].name}/dataset.tar")
    else:
        print("Paket final belum terkonfirmasi di Drive.")


if ATTEMPT:
    view_summary(ATTEMPT)
else:
    print("Belum ada run pada sesi ini.")